# NB68: Supply Chain Traceability

Kafka -> Spark -> Redis/MinIO

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q "numpy<2.0.0" findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Redis**
- **MinIO**

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start
# Start MinIO
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!mkdir -p /content/minio_data
!MINIO_ROOT_USER=minioadmin MINIO_ROOT_PASSWORD=minioadmin ./minio server /content/minio_data --console-address ":9001" &> minio.log &

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(9000) # MinIO
time.sleep(5) # Extra buffer for MinIO
wait_for_port(6379) # Redis


## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (RFID)

Simulates package movement.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting RFID Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 100 tags...")
for i in range(100):
    data = {'rfid': f'tag_{i}', 'loc': 'warehouse'}
    producer.send('input-topic', json.dumps(data).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Traceability Spark Job

Updates current location of each item in Redis.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis
import json

spark = SparkSession.builder.appName("Supply").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    r = redis.Redis()
    for row in rows:
        item = json.loads(row.value)
        r.set(f"loc:{item['rfid']}", item['loc'])
    print(f"Batch {epoch_id} updated locations.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check locations in Redis.

In [ ]:
import redis
r = redis.Redis()
keys = r.keys("loc:*")
print(f"Tracked Items: {len(keys)}")